# 10 - SAINT | Sensor 3

Full SAINT with intersample attention. Settings tuned: lr=1e-4, 3 layers, early stopping.

Trained from scratch on this sensor. 70/15/15 split; validation picks the best epoch, test used once.

### Cell 1 - Imports

In [1]:
import os
import copy
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

### Cell 2 - Device

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

Device : cuda
GPU    : NVIDIA GeForce RTX 3050 6GB Laptop GPU


### Cell 3 - Random seed

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

### Cell 4 - Load Sensor 3

In [4]:
df = pd.read_csv("../data/processed/sensor3_processed.csv")
print(df.shape)
df.head()

(3885792, 15)


,Dur,Proto,Sport,Dir,Dport,State,sTos,dTos,TotPkts,TotBytes,SrcBytes,Hour,Minute,Second,ActivityLabel
0,-0.382178,2.0,119.0,0.0,13395.0,344.0,-0.016359,-0.012325,-0.017028,-0.014875,-0.014910,-1.373052,-1.686039,-1.680628,1
1,-0.382031,13.0,58403.0,3.0,68168.0,17.0,-0.016359,-0.012325,-0.016801,-0.014889,-0.014960,-1.373052,-1.686039,-1.680628,0
2,3.578163,12.0,46643.0,5.0,54926.0,183.0,-0.016359,-0.012325,0.041651,-0.010092,-0.011670,-1.373052,-1.686039,-1.680628,0
3,3.606046,6.0,73653.0,0.0,81768.0,158.0,-0.016359,-0.012325,0.111431,-0.001854,0.007483,-1.373052,-1.686039,-1.680628,0
4,3.464072,12.0,71626.0,5.0,65058.0,43.0,-0.016359,-0.012325,0.060908,0.021488,0.000964,-1.373052,-1.686039,-1.680628,0


### Cell 5 - Features and labels

In [5]:
TARGET = "ActivityLabel"

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

print("Features :", X.shape[1])
print("Samples  :", X.shape[0])
print("Classes  :", y.nunique())
display(y.value_counts())

Features : 14
Samples  : 3885792
Classes  : 2


### Cell 6 - Encode labels

In [6]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

NUM_CLASSES = len(label_encoder.classes_)
print("Number of classes:", NUM_CLASSES)

Number of classes: 2


### Cell 7 - Train / validation / test split

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train :", X_train.shape)
print("Valid :", X_val.shape)
print("Test  :", X_test.shape)

Train : (2720054, 14)
Valid : (582869, 14)
Test  : (582869, 14)


### Cell 8 - Scale features (fit on train only)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val   = scaler.transform(X_val).astype("float32")
X_test  = scaler.transform(X_test).astype("float32")

print(X_train.dtype, X_train.shape)

float32 (2720054, 14)


### Cell 9 - PyTorch Dataset

In [9]:
class SensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

### Cell 10 - DataLoaders

In [10]:
BATCH_SIZE = 512

train_dataset = SensorDataset(X_train, y_train)
val_dataset   = SensorDataset(X_val,   y_val)
test_dataset  = SensorDataset(X_test,  y_test)

# drop_last=True keeps intersample-attention batch sizes consistent.
# On Windows + Jupyter, if it hangs, set num_workers=0.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))

Train batches: 5312


### Cell 11 - Model config
Dimensions derived from data. 3 layers, tuned during Sensor 1 debugging.

In [11]:
NUM_FEATURES = X_train.shape[1]
NUM_TOKENS   = NUM_FEATURES + 1

EMBED_DIM  = 64
NUM_HEADS  = 8
NUM_LAYERS = 3
FFN_DIM    = 256
DROPOUT    = 0.2

assert EMBED_DIM % NUM_HEADS == 0
print("Features:", NUM_FEATURES, "| Classes:", NUM_CLASSES)

Features: 14 | Classes: 2


### Cell 12 - Numerical feature tokenizer

In [12]:
class NumericalFeatureTokenizer(nn.Module):
    def __init__(self, num_features, embed_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_features, embed_dim))
        self.bias   = nn.Parameter(torch.randn(num_features, embed_dim))

    def forward(self, x):
        x = x.unsqueeze(-1)
        return x * self.weight + self.bias

### Cell 13 - Feed forward network

In [13]:
class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

### Cell 14 - SAINT block (column self-attention + intersample attention)

In [14]:
class SAINTBlock(nn.Module):
    def __init__(self, dim, num_tokens, heads, ff_dim, dropout):
        super().__init__()
        # Column (feature) self-attention
        self.col_norm    = nn.LayerNorm(dim)
        self.col_attn    = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.col_ff_norm = nn.LayerNorm(dim)
        self.col_ff      = FeedForward(dim, ff_dim, dropout)

        # Row (intersample) attention over flattened per-sample tokens
        row_dim = num_tokens * dim
        self.row_norm    = nn.LayerNorm(dim)
        self.row_attn    = nn.MultiheadAttention(row_dim, heads, dropout=dropout, batch_first=True)
        self.row_ff_norm = nn.LayerNorm(dim)
        self.row_ff      = FeedForward(dim, ff_dim, dropout)

    def forward(self, x):
        h = self.col_norm(x)
        a, _ = self.col_attn(h, h, h, need_weights=False)
        x = x + a
        x = x + self.col_ff(self.col_ff_norm(x))

        b, n, d = x.shape
        h = self.row_norm(x).reshape(1, b, n * d)
        a, _ = self.row_attn(h, h, h, need_weights=False)
        x = x + a.reshape(b, n, d)
        x = x + self.row_ff(self.row_ff_norm(x))
        return x

### Cell 15 - SAINT model

In [15]:
class SAINT(nn.Module):
    def __init__(self, num_features, num_classes, embed_dim,
                 num_heads, num_layers, ff_dim, dropout=0.2):
        super().__init__()
        num_tokens = num_features + 1

        self.tokenizer = NumericalFeatureTokenizer(num_features, embed_dim)
        self.feature_embedding = nn.Parameter(torch.randn(1, num_features, embed_dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        self.encoder = nn.ModuleList([
            SAINTBlock(embed_dim, num_tokens, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, num_classes),
        )
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.cls_token, std=0.02)
        nn.init.normal_(self.feature_embedding, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.tokenizer(x)
        x = x + self.feature_embedding

        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.dropout(x)

        for block in self.encoder:
            x = block(x)

        x = self.final_norm(x)
        cls_representation = x[:, 0]
        return self.head(cls_representation)

### Cell 16 - Build model

In [16]:
model = SAINT(
    num_features=NUM_FEATURES,
    num_classes=NUM_CLASSES,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FFN_DIM,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print("Parameters:", f"{n_params:,}")

Parameters: 11,340,738


### Cell 17 - Loss, optimizer, scheduler
lr=1e-4 (1e-3 was unstable for the row attention). Remove class weights if the sensor is balanced.

In [17]:
EPOCHS = 30

counts  = np.bincount(y_train, minlength=NUM_CLASSES)
weights = torch.tensor(counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

### Cell 18 - Epoch runner

In [18]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    torch.set_grad_enabled(train)
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        if train:
            optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        n += labels.size(0)
    torch.set_grad_enabled(True)
    return total_loss / n, correct / n

### Cell 19 - Train loop (early stopping, keeps best-val weights)

In [19]:
best_val_acc, best_state = 0.0, None
patience, wait = 6, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    torch.set_grad_enabled(True)      # guards against a stale no_grad leaking in
    tr_loss, tr_acc, seen = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(inputs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * labels.size(0)
        tr_acc  += (out.argmax(1) == labels).sum().item()
        seen    += labels.size(0)

    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc, best_state, wait = val_acc, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1

    print(f"Epoch {epoch:03d} | train {tr_loss/seen:.4f}/{tr_acc/seen:.4f} | "
          f"val {val_loss:.4f}/{val_acc:.4f}")

    if wait >= patience:
        print("Early stop at epoch", epoch)
        break

print("Best val acc:", round(best_val_acc, 4))

Epoch 001 | train 0.2498/0.8659 | val 0.2413/0.8608
Epoch 002 | train 0.2119/0.8832 | val 0.2097/0.8882
Epoch 003 | train 0.1974/0.8941 | val 0.1831/0.9042
Epoch 004 | train 0.1882/0.8996 | val 0.1885/0.9017
Epoch 005 | train 0.1818/0.9035 | val 0.1743/0.9095
Epoch 006 | train 0.1767/0.9061 | val 0.1765/0.9086
Epoch 007 | train 0.1720/0.9085 | val 0.1683/0.9197
Epoch 008 | train 0.1685/0.9100 | val 0.1819/0.9113
Epoch 009 | train 0.1649/0.9118 | val 0.1637/0.9233
Epoch 010 | train 0.1620/0.9135 | val 0.1641/0.9142
Epoch 011 | train 0.1594/0.9151 | val 0.1585/0.9172
Epoch 012 | train 0.1567/0.9161 | val 0.1615/0.9073
Epoch 013 | train 0.1545/0.9172 | val 0.1598/0.9291
Epoch 014 | train 0.1523/0.9184 | val 0.2077/0.9198
Epoch 015 | train 0.1498/0.9196 | val 0.1539/0.9177
Epoch 016 | train 0.1474/0.9207 | val 0.1623/0.9237
Epoch 017 | train 0.1453/0.9223 | val 0.1518/0.9189
Epoch 018 | train 0.1432/0.9230 | val 0.1838/0.9256
Epoch 019 | train 0.1412/0.9241 | val 0.1806/0.9221
Early stop a

### Cell 20 - Predict on test set

In [20]:
model.load_state_dict(best_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs.to(DEVICE))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

### Cell 21 - Accuracy, precision, recall, F1

In [21]:
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
rec  = recall_score(all_labels, all_preds, average="weighted", zero_division=0)
f1_w = f1_score(all_labels, all_preds, average="weighted", zero_division=0)
f1_m = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"Accuracy      : {acc:.4f}")
print(f"Precision (w) : {prec:.4f}")
print(f"Recall (w)    : {rec:.4f}")
print(f"F1 (weighted) : {f1_w:.4f}")
print(f"F1 (macro)    : {f1_m:.4f}")

Accuracy      : 0.9292
Precision (w) : 0.9594
Recall (w)    : 0.9292
F1 (weighted) : 0.9384
F1 (macro)    : 0.8152


### Cell 22 - Classification report

In [22]:
target_names = [str(c) for c in label_encoder.classes_]
print(classification_report(all_labels, all_preds,
                            target_names=target_names, digits=4))

              precision    recall  f1-score   support

           0     0.9956    0.9275    0.9604    538769
           1     0.5176    0.9500    0.6701     44100

    accuracy                         0.9292    582869
   macro avg     0.7566    0.9388    0.8152    582869
weighted avg     0.9594    0.9292    0.9384    582869



### Cell 23 - Confusion matrix

In [23]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("SAINT - Sensor 3 Confusion Matrix")
plt.tight_layout()
plt.show()